In [1]:
import pandas as pd
import seaborn as sn
from dataclasses import dataclass

from src.summarised_result import SummarisedResultSettings, reshape_group_additional

# Visualising Incidence and Prevalence results

The plots you get out of the IncidencePrevalence R package are nice.
Let's try to copy them in Python.

First, we need our data.
I have some outputs from the R package in the demo data.

In [2]:
incidence = pd.read_csv("demo-data/incidence.csv")
point_prevalence = pd.read_csv("demo-data/pointPrevalence.csv")
period_prevalence = pd.read_csv("demo-data/periodPrevalence.csv")

## Incidence (part 1)
First, we need to check what the incidence data look like.

In [3]:
print(incidence.shape)
incidence.head()

(677, 13)


,result_id,cdm_name,group_name,group_level,strata_name,strata_level,variable_name,variable_level,estimate_name,estimate_type,estimate_value,additional_name,additional_level
0,1,postgres_omop,denominator_cohort_name &&& outcome_cohort_name,denominator_cohort_1 &&& neoplasm,overall,overall,Denominator,NaN,denominator_count,integer,10,incidence_start_date &&& incidence_end_date &&...,1927-01-01 &&& 1927-12-31 &&& years
1,1,postgres_omop,denominator_cohort_name &&& outcome_cohort_name,denominator_cohort_1 &&& neoplasm,overall,overall,Outcome,NaN,outcome_count,integer,0,incidence_start_date &&& incidence_end_date &&...,1927-01-01 &&& 1927-12-31 &&& years
2,1,postgres_omop,denominator_cohort_name &&& outcome_cohort_name,denominator_cohort_1 &&& neoplasm,overall,overall,Denominator,NaN,person_days,numeric,2378,incidence_start_date &&& incidence_end_date &&...,1927-01-01 &&& 1927-12-31 &&& years
3,1,postgres_omop,denominator_cohort_name &&& outcome_cohort_name,denominator_cohort_1 &&& neoplasm,overall,overall,Denominator,NaN,person_years,numeric,6.511,incidence_start_date &&& incidence_end_date &&...,1927-01-01 &&& 1927-12-31 &&& years
4,1,postgres_omop,denominator_cohort_name &&& outcome_cohort_name,denominator_cohort_1 &&& neoplasm,overall,overall,Outcome,NaN,incidence_100000_pys,numeric,0,incidence_start_date &&& incidence_end_date &&...,1927-01-01 &&& 1927-12-31 &&& years


They have done a slightly strange thing, I think to coerce the data into the summarised_result format.
When you `importSummarisedResult` in R, you get out:

```
Rows: 85
Columns: 27
$ X                                    <int> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, …
$ cdm_name                             <chr> "postgres_omop", "postgres_omop", "postgres_omop", …
$ denominator_cohort_name              <chr> "denominator_cohort_1", "denominator_cohort_1", "de…
$ outcome_cohort_name                  <chr> "neoplasm", "neoplasm", "neoplasm", "neoplasm", "ne…
$ incidence_start_date                 <date> 1927-01-01, 1928-01-01, 1929-01-01, 1930-01-01, 19…
$ incidence_end_date                   <date> 1927-12-31, 1928-12-31, 1929-12-31, 1930-12-31, 19…
$ analysis_interval                    <chr> "years", "years", "years", "years", "years", "years…
$ analysis_censor_cohort_name          <chr> "None", "None", "None", "None", "None", "None", "No…
$ analysis_complete_database_intervals <lgl> TRUE, TRUE, TRUE, TRUE, TRUE, TRUE, TRUE, TRUE, TRU…
$ analysis_outcome_washout             <int> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, …
$ analysis_repeated_events             <lgl> FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FA…
$ denominator_age_group                <chr> "0 to 150", "0 to 150", "0 to 150", "0 to 150", "0 …
$ denominator_days_prior_observation   <int> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, …
$ denominator_end_date                 <date> 2100-01-01, 2100-01-01, 2100-01-01, 2100-01-01, 21…
$ denominator_requirements_at_entry    <lgl> FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FA…
$ denominator_sex                      <chr> "Both", "Both", "Both", "Both", "Both", "Both", "Bo…
$ denominator_start_date               <date> 1900-01-01, 1900-01-01, 1900-01-01, 1900-01-01, 19…
$ denominator_target_cohort_name       <chr> "None", "None", "None", "None", "None", "None", "No…
$ denominator_time_at_risk             <chr> "0 to Inf", "0 to Inf", "0 to Inf", "0 to Inf", "0 …
$ denominator_count                    <int> 10, 20, 31, 34, 44, 57, 69, 80, 88, 99, 110, 117, 1…
$ outcome_count                        <int> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, …
$ person_days                          <int> 2378, 5333, 9354, 12022, 14349, 18342, 23091, 27299…
$ person_years                         <dbl> 6.511, 14.601, 25.610, 32.914, 39.285, 50.218, 63.2…
$ incidence_100000_pys                 <dbl> 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.…
$ incidence_100000_pys_95CI_lower      <dbl> 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.…
$ incidence_100000_pys_95CI_upper      <dbl> 56656.112, 25264.567, 14404.059, 11207.630, 9390.04…
$ result_type                          <chr> "tidy_incidence", "tidy_incidence", "tidy_incidence…
```

So the number of columns has to slightly more than double, and the number of rows has to go down by a factor of seven.
Interesting.

If you look at the `group_name` column, it contains what are two column names in the R table: `denominator_cohort_name` and `outcome_cohort_name`.
The levels of these columns are stored in `group_level`.
There is a similar thing going on with `additional_name` and `additional_level`.

Maybe the thing to do will be to just look at the results where `result_id` is 1.

In [4]:
incidence.loc[incidence["result_id"] == 1]

,result_id,cdm_name,group_name,group_level,strata_name,strata_level,variable_name,variable_level,estimate_name,estimate_type,estimate_value,additional_name,additional_level
0,1,postgres_omop,denominator_cohort_name &&& outcome_cohort_name,denominator_cohort_1 &&& neoplasm,overall,overall,Denominator,NaN,denominator_count,integer,10,incidence_start_date &&& incidence_end_date &&...,1927-01-01 &&& 1927-12-31 &&& years
1,1,postgres_omop,denominator_cohort_name &&& outcome_cohort_name,denominator_cohort_1 &&& neoplasm,overall,overall,Outcome,NaN,outcome_count,integer,0,incidence_start_date &&& incidence_end_date &&...,1927-01-01 &&& 1927-12-31 &&& years
2,1,postgres_omop,denominator_cohort_name &&& outcome_cohort_name,denominator_cohort_1 &&& neoplasm,overall,overall,Denominator,NaN,person_days,numeric,2378,incidence_start_date &&& incidence_end_date &&...,1927-01-01 &&& 1927-12-31 &&& years
3,1,postgres_omop,denominator_cohort_name &&& outcome_cohort_name,denominator_cohort_1 &&& neoplasm,overall,overall,Denominator,NaN,person_years,numeric,6.511,incidence_start_date &&& incidence_end_date &&...,1927-01-01 &&& 1927-12-31 &&& years
4,1,postgres_omop,denominator_cohort_name &&& outcome_cohort_name,denominator_cohort_1 &&& neoplasm,overall,overall,Outcome,NaN,incidence_100000_pys,numeric,0,incidence_start_date &&& incidence_end_date &&...,1927-01-01 &&& 1927-12-31 &&& years
...,...,...,...,...,...,...,...,...,...,...,...,...,...
653,1,NaN,overall,overall,overall,overall,settings,NaN,denominator_requirements_at_entry,character,FALSE,overall,overall
654,1,NaN,overall,overall,overall,overall,settings,NaN,denominator_sex,character,Both,overall,overall
655,1,NaN,overall,overall,overall,overall,settings,NaN,denominator_start_date,character,1900-01-01,overall,overall
656,1,NaN,overall,overall,overall,overall,settings,NaN,denominator_target_cohort_name,character,NaN,overall,overall


Well it's not that.

I'm pretty sure what needs to happen is that the `group_name` has to be split on `"&&&"`, as does `group_level`, and the values of the column names in `group_name` come from `group_level` and the same with `additional_*`.
Then, the `variable_name`, `estimate_name` and `estimate_value` are used in a pivot-y way to reshape the table.

Perhaps I need to read the code for `importSummarisedResult` to get the rules out.

## importSummarisedResult

The code for `importSummarisedResult` is fairly short.
Unfortunately that's because the logic I want isn't there, but in `newSummarisedResult`, which is much more complicated.

```mermaid
graph TD
    importSummarisedResult --> newSummarisedResult
    newSummarisedResult --> constructSummarisedResult
    newSummarisedResult --> validateSummarisedResult
    constructSummarisedResult --> createSettings
    validateSummarisedResult --> validateResultSettings
    validateSummarisedResult --> validateSummarisedResultTable
```

I think I can actually just ignore most of that; I just need to have a bash at it, see if it matches the incidence table.

What helps is that I now know you can read stuff about parsing the table from fields in the table itself.


### Parsing settings

The tables store their settings where the `variable_name` is `"settings"`.

In [5]:
incidence.loc[incidence["variable_name"] == "settings"]

,result_id,cdm_name,group_name,group_level,strata_name,strata_level,variable_name,variable_level,estimate_name,estimate_type,estimate_value,additional_name,additional_level
639,1,NaN,overall,overall,overall,overall,settings,NaN,result_type,character,incidence,overall,overall
640,1,NaN,overall,overall,overall,overall,settings,NaN,package_name,character,IncidencePrevalence,overall,overall
641,1,NaN,overall,overall,overall,overall,settings,NaN,package_version,character,1.2.1,overall,overall
642,1,NaN,overall,overall,overall,overall,settings,NaN,group,character,denominator_cohort_name &&& outcome_cohort_name,overall,overall
643,1,NaN,overall,overall,overall,overall,settings,NaN,strata,character,NaN,overall,overall
644,1,NaN,overall,overall,overall,overall,settings,NaN,additional,character,incidence_start_date &&& incidence_end_date &&...,overall,overall
645,1,NaN,overall,overall,overall,overall,settings,NaN,min_cell_count,character,5,overall,overall
646,1,NaN,overall,overall,overall,overall,settings,NaN,analysis_censor_cohort_name,character,NaN,overall,overall
647,1,NaN,overall,overall,overall,overall,settings,NaN,analysis_complete_database_intervals,character,TRUE,overall,overall
648,1,NaN,overall,overall,overall,overall,settings,NaN,analysis_outcome_washout,character,0,overall,overall


I think what I need then is a `SummarisedResultSettings` class to work with.
The code for this is part of the package in `src/summarised_result.py`.

In [6]:
SummarisedResultSettings.from_table(incidence.loc[(incidence["result_id"] == 1)])

SummarisedResultSettings(result_type='incidence', package_name='IncidencePrevalence', package_version=SemanticVersion(major=1, minor=2, patch=1), groups=['denominator_cohort_name', 'outcome_cohort_name'], strata=None, additional=['incidence_start_date', 'incidence_end_date', 'analysis_interval'], min_cell_count=5, variables=['strata', 'analysis_censor_cohort_name', 'analysis_complete_database_intervals', 'analysis_outcome_washout', 'analysis_repeated_events', 'denominator_age_group', 'denominator_days_prior_observation', 'denominator_end_date', 'denominator_requirements_at_entry', 'denominator_sex', 'denominator_start_date', 'denominator_target_cohort_name', 'denominator_time_at_risk'])

## Exploding rows

To show how I would normally do something like this, I'll take the first two rows to test on:

In [7]:
test_exploding_row = incidence.iloc[0:2]
test_exploding_row

,result_id,cdm_name,group_name,group_level,strata_name,strata_level,variable_name,variable_level,estimate_name,estimate_type,estimate_value,additional_name,additional_level
0,1,postgres_omop,denominator_cohort_name &&& outcome_cohort_name,denominator_cohort_1 &&& neoplasm,overall,overall,Denominator,NaN,denominator_count,integer,10,incidence_start_date &&& incidence_end_date &&...,1927-01-01 &&& 1927-12-31 &&& years
1,1,postgres_omop,denominator_cohort_name &&& outcome_cohort_name,denominator_cohort_1 &&& neoplasm,overall,overall,Outcome,NaN,outcome_count,integer,0,incidence_start_date &&& incidence_end_date &&...,1927-01-01 &&& 1927-12-31 &&& years


Normally I would split the columns containing lists

In [8]:
test_exploding_row["group_name"].str.split(" &&& ")

0    [denominator_cohort_name, outcome_cohort_name]
1    [denominator_cohort_name, outcome_cohort_name]
Name: group_name, dtype: object

then use `.explode` to make multiple rows:

In [9]:
test_exploding_row["group_name"] = test_exploding_row["group_name"].str.split(" &&& ")
test_exploding_row["group_level"] = test_exploding_row["group_level"].str.split(" &&& ")

exploded_group = test_exploding_row.explode(
    ["group_name", "group_level"]
)

exploded_group

,result_id,cdm_name,group_name,group_level,strata_name,strata_level,variable_name,variable_level,estimate_name,estimate_type,estimate_value,additional_name,additional_level
0,1,postgres_omop,denominator_cohort_name,denominator_cohort_1,overall,overall,Denominator,NaN,denominator_count,integer,10,incidence_start_date &&& incidence_end_date &&...,1927-01-01 &&& 1927-12-31 &&& years
0,1,postgres_omop,outcome_cohort_name,neoplasm,overall,overall,Denominator,NaN,denominator_count,integer,10,incidence_start_date &&& incidence_end_date &&...,1927-01-01 &&& 1927-12-31 &&& years
1,1,postgres_omop,denominator_cohort_name,denominator_cohort_1,overall,overall,Outcome,NaN,outcome_count,integer,0,incidence_start_date &&& incidence_end_date &&...,1927-01-01 &&& 1927-12-31 &&& years
1,1,postgres_omop,outcome_cohort_name,neoplasm,overall,overall,Outcome,NaN,outcome_count,integer,0,incidence_start_date &&& incidence_end_date &&...,1927-01-01 &&& 1927-12-31 &&& years


In [10]:
exploded_group["additional_name"] = test_exploding_row["additional_name"].str.split(" &&& ")
exploded_group["additional_level"] = test_exploding_row["additional_level"].str.split(" &&& ")

fully_exploded_example = exploded_group.explode(["additional_name", "additional_level"]).reset_index()
fully_exploded_example

,index,result_id,cdm_name,group_name,group_level,strata_name,strata_level,variable_name,variable_level,estimate_name,estimate_type,estimate_value,additional_name,additional_level
0,0,1,postgres_omop,denominator_cohort_name,denominator_cohort_1,overall,overall,Denominator,NaN,denominator_count,integer,10,incidence_start_date,1927-01-01
1,0,1,postgres_omop,denominator_cohort_name,denominator_cohort_1,overall,overall,Denominator,NaN,denominator_count,integer,10,incidence_end_date,1927-12-31
2,0,1,postgres_omop,denominator_cohort_name,denominator_cohort_1,overall,overall,Denominator,NaN,denominator_count,integer,10,analysis_interval,years
3,0,1,postgres_omop,outcome_cohort_name,neoplasm,overall,overall,Denominator,NaN,denominator_count,integer,10,incidence_start_date,1927-01-01
4,0,1,postgres_omop,outcome_cohort_name,neoplasm,overall,overall,Denominator,NaN,denominator_count,integer,10,incidence_end_date,1927-12-31
5,0,1,postgres_omop,outcome_cohort_name,neoplasm,overall,overall,Denominator,NaN,denominator_count,integer,10,analysis_interval,years
6,1,1,postgres_omop,denominator_cohort_name,denominator_cohort_1,overall,overall,Outcome,NaN,outcome_count,integer,0,incidence_start_date,1927-01-01
7,1,1,postgres_omop,denominator_cohort_name,denominator_cohort_1,overall,overall,Outcome,NaN,outcome_count,integer,0,incidence_end_date,1927-12-31
8,1,1,postgres_omop,denominator_cohort_name,denominator_cohort_1,overall,overall,Outcome,NaN,outcome_count,integer,0,analysis_interval,years
9,1,1,postgres_omop,outcome_cohort_name,neoplasm,overall,overall,Outcome,NaN,outcome_count,integer,0,incidence_start_date,1927-01-01


Now we have 6 rows per original row!
Can we simply pivot on those?

In [17]:
group_example = fully_exploded_example[["group_name", "group_level"]].pivot(
    columns="group_name", values="group_level"
)
group_example

group_name,denominator_cohort_name,outcome_cohort_name
0,denominator_cohort_1,NaN
1,denominator_cohort_1,NaN
2,denominator_cohort_1,NaN
3,NaN,neoplasm
4,NaN,neoplasm
5,NaN,neoplasm
6,denominator_cohort_1,NaN
7,denominator_cohort_1,NaN
8,denominator_cohort_1,NaN
9,NaN,neoplasm


In [16]:
additional_example = fully_exploded_example[["additional_name", "additional_level"]].pivot(
    columns="additional_name", values="additional_level"
)
additional_example

additional_name,analysis_interval,incidence_end_date,incidence_start_date
0,NaN,NaN,1927-01-01
1,NaN,1927-12-31,NaN
2,years,NaN,NaN
3,NaN,NaN,1927-01-01
4,NaN,1927-12-31,NaN
5,years,NaN,NaN
6,NaN,NaN,1927-01-01
7,NaN,1927-12-31,NaN
8,years,NaN,NaN
9,NaN,NaN,1927-01-01


In [13]:
fully_exploded_example.join(
    group_example
).join(
    additional_example
).drop(
    ["group_name", "group_level", "additional_name", "additional_level"], axis=1
).set_index("index")

,result_id,cdm_name,strata_name,strata_level,variable_name,variable_level,estimate_name,estimate_type,estimate_value,denominator_cohort_name,outcome_cohort_name,analysis_interval,incidence_end_date,incidence_start_date
index,,,,,,,,,,,,,,
0,1,postgres_omop,overall,overall,Denominator,NaN,denominator_count,integer,10,denominator_cohort_1,NaN,NaN,NaN,1927-01-01
0,1,postgres_omop,overall,overall,Denominator,NaN,denominator_count,integer,10,denominator_cohort_1,NaN,NaN,1927-12-31,NaN
0,1,postgres_omop,overall,overall,Denominator,NaN,denominator_count,integer,10,denominator_cohort_1,NaN,years,NaN,NaN
0,1,postgres_omop,overall,overall,Denominator,NaN,denominator_count,integer,10,NaN,neoplasm,NaN,NaN,1927-01-01
0,1,postgres_omop,overall,overall,Denominator,NaN,denominator_count,integer,10,NaN,neoplasm,NaN,1927-12-31,NaN
0,1,postgres_omop,overall,overall,Denominator,NaN,denominator_count,integer,10,NaN,neoplasm,years,NaN,NaN
1,1,postgres_omop,overall,overall,Outcome,NaN,outcome_count,integer,0,denominator_cohort_1,NaN,NaN,NaN,1927-01-01
1,1,postgres_omop,overall,overall,Outcome,NaN,outcome_count,integer,0,denominator_cohort_1,NaN,NaN,1927-12-31,NaN
1,1,postgres_omop,overall,overall,Outcome,NaN,outcome_count,integer,0,denominator_cohort_1,NaN,years,NaN,NaN


Right, this is nearly it and I'll leave that cell because it shows how it gets built, but I think we can actually join that back to the original table instead.

We can take the exploded rows, join them to the original table's index, then take the first non-`NaN` value for each column to get the target columns for the original table.

In [14]:
additional_example.join(
    group_example
).join(
    fully_exploded_example["index"]
).set_index("index").groupby(level=0).first()

,analysis_interval,incidence_end_date,incidence_start_date,denominator_cohort_name,outcome_cohort_name
index,,,,,
0,years,1927-12-31,1927-01-01,denominator_cohort_1,neoplasm
1,years,1927-12-31,1927-01-01,denominator_cohort_1,neoplasm


In [18]:
partly_reshaped_example = reshape_group_additional(
    incidence.loc[
        ~(incidence["variable_name"] == "settings") & (incidence["result_id"] == 1)
    ]
)

partly_reshaped_example

,result_id,cdm_name,strata_name,strata_level,variable_name,variable_level,estimate_name,estimate_type,estimate_value,denominator_cohort_name,outcome_cohort_name,analysis_interval,incidence_end_date,incidence_start_date
0,1,postgres_omop,overall,overall,Denominator,NaN,denominator_count,integer,10,denominator_cohort_1,neoplasm,years,1927-12-31,1927-01-01
1,1,postgres_omop,overall,overall,Outcome,NaN,outcome_count,integer,0,denominator_cohort_1,neoplasm,years,1927-12-31,1927-01-01
2,1,postgres_omop,overall,overall,Denominator,NaN,person_days,numeric,2378,denominator_cohort_1,neoplasm,years,1927-12-31,1927-01-01
3,1,postgres_omop,overall,overall,Denominator,NaN,person_years,numeric,6.511,denominator_cohort_1,neoplasm,years,1927-12-31,1927-01-01
4,1,postgres_omop,overall,overall,Outcome,NaN,incidence_100000_pys,numeric,0,denominator_cohort_1,neoplasm,years,1927-12-31,1927-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590,1,postgres_omop,overall,overall,Denominator,NaN,person_days,numeric,7244178,denominator_cohort_1,neoplasm,years,2011-12-31,2011-01-01
591,1,postgres_omop,overall,overall,Denominator,NaN,person_years,numeric,19833.478,denominator_cohort_1,neoplasm,years,2011-12-31,2011-01-01
592,1,postgres_omop,overall,overall,Outcome,NaN,incidence_100000_pys,numeric,0,denominator_cohort_1,neoplasm,years,2011-12-31,2011-01-01
593,1,postgres_omop,overall,overall,Outcome,NaN,incidence_100000_pys_95CI_lower,numeric,0,denominator_cohort_1,neoplasm,years,2011-12-31,2011-01-01


This gets us most of the way there.
The next part is to reshape it in a similar way with estimate_name, estimate_type, and estimate_value.

In [68]:
estimate_pivot = partly_reshaped_example.pivot(
    columns="estimate_name",
    values="estimate_value"
)
estimate_pivot

estimate_name,denominator_count,incidence_100000_pys,incidence_100000_pys_95CI_lower,incidence_100000_pys_95CI_upper,outcome_count,person_days,person_years
0,10,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,0,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,2378,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,6.511
4,NaN,0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
590,NaN,NaN,NaN,NaN,NaN,7244178,NaN
591,NaN,NaN,NaN,NaN,NaN,NaN,19833.478
592,NaN,0,NaN,NaN,NaN,NaN,NaN
593,NaN,NaN,0,NaN,NaN,NaN,NaN


This means we have a table with a bunch of `NaN` again.

In [90]:
no_estimate_gunk = partly_reshaped_example.drop(
    ["estimate_name", "estimate_type", "estimate_value", "variable_level"],
    axis=1
)

group_columns = list(no_estimate_gunk.columns)

no_estimate_gunk.join(
    estimate_pivot
).groupby(group_columns).first().reset_index()

,result_id,cdm_name,strata_name,strata_level,variable_name,denominator_cohort_name,outcome_cohort_name,analysis_interval,incidence_end_date,incidence_start_date,denominator_count,incidence_100000_pys,incidence_100000_pys_95CI_lower,incidence_100000_pys_95CI_upper,outcome_count,person_days,person_years
0,1,postgres_omop,overall,overall,Denominator,denominator_cohort_1,neoplasm,years,1927-12-31,1927-01-01,10,NaN,NaN,NaN,NaN,2378,6.511
1,1,postgres_omop,overall,overall,Denominator,denominator_cohort_1,neoplasm,years,1928-12-31,1928-01-01,20,NaN,NaN,NaN,NaN,5333,14.601
2,1,postgres_omop,overall,overall,Denominator,denominator_cohort_1,neoplasm,years,1929-12-31,1929-01-01,31,NaN,NaN,NaN,NaN,9354,25.61
3,1,postgres_omop,overall,overall,Denominator,denominator_cohort_1,neoplasm,years,1930-12-31,1930-01-01,34,NaN,NaN,NaN,NaN,12022,32.914
4,1,postgres_omop,overall,overall,Denominator,denominator_cohort_1,neoplasm,years,1931-12-31,1931-01-01,44,NaN,NaN,NaN,NaN,14349,39.285
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,1,postgres_omop,overall,overall,Outcome,denominator_cohort_1,neoplasm,years,2007-12-31,2007-01-01,NaN,0,0,7.895,0,NaN,NaN
166,1,postgres_omop,overall,overall,Outcome,denominator_cohort_1,neoplasm,years,2008-12-31,2008-01-01,NaN,0,0,7.846,0,NaN,NaN
167,1,postgres_omop,overall,overall,Outcome,denominator_cohort_1,neoplasm,years,2009-12-31,2009-01-01,NaN,0,0,8.026,0,NaN,NaN
168,1,postgres_omop,overall,overall,Outcome,denominator_cohort_1,neoplasm,years,2010-12-31,2010-01-01,NaN,0,0,8.725,0,NaN,NaN


This gets us nearly there, though we still have twice as many rows as we need.